### Notebook scope

### Purpose
Plot O3, U60N10, and U60N50 evolution for available hindcast cases.

### Scientific question
Do O3 pathway differences co-evolve with vortex wind pathway differences across cases?

### Method
Use cleaned O3 partial column and cached/raw U interpolated to 10 and 50 hPa at 60N. Reference line is BWCN same year.

### Expected output
Per-case evolution figures plus overview CSV.

### Interpretation
O3 spread should be interpreted together with U60N10/U60N50 spread and final warming timing.

### Caveat
Computing U60 from raw U can be slow the first time; values are cached under outputs/cache.


### Setup imports and shared paths

### Purpose
Load the shared Extention_analysis utility module and create output directories.

### Scientific question
Can every notebook start from a clean kernel and still find the same data roots and output roots?

### Method
Use DATA_ROOT, HINDCAST_ROOT, BWCN_ROOT, B2000_ROOT, WORK_ROOT, FIG_DIR, TAB_DIR, CACHE_DIR, and LOG_DIR from hindcast_extension_utils.py. No diagnostics are calculated in this cell.

### Expected output
Printed path check only. No figure is generated by this setup cell.

### Interpretation
Successful execution means the notebook can access the common utilities and write outputs in the agreed directory tree.

### Caveat
This setup does not prove that every downstream data product exists; missing products are logged later.


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

import hindcast_extension_utils as hx
from hindcast_extension_utils import *
_assign_member_short = hx._assign_member_short

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)

### Create multi-case evolution plots

### Purpose
Loop through available cases and save one O3/U figure per case plus a summary table.

### Scientific question
Which cases show synchronized ozone loss and polar-vortex wind evolution?

### Method
Thin lines are members; thick line is ensemble mean; shading is +/-1 ensemble standard deviation. Reference O3 and U are loaded for the same BWCN year when available.

### Expected output
outputs/figures/02_evolution_<case>_O3_U60N10_U60N50.png/pdf and outputs/tables/02_multicase_evolution_summary.csv.

### Interpretation
Large U spread before O3 spread supports a vortex-pathway interpretation of ozone uncertainty.

### Caveat
If U files are missing or too incomplete, the case is skipped for wind panels and logged.


In [ ]:
inv = discover_hindcast_cases()
summary_rows = []

def plot_member_envelope(ax, da, date, label, color, ref_da=None, ref_date=None):
    if da is None:
        ax.text(0.5, 0.5, f"missing {label}", transform=ax.transAxes, ha="center", va="center")
        return
    x = date_to_doy(date) if np.asarray(date).max() > 1000 else np.arange(da.sizes["lead_time"])
    vals = da.values
    for row in vals:
        ax.plot(x[:len(row)], row, color=color, alpha=0.18, lw=0.7)
    mean = np.nanmean(vals, axis=0); std = np.nanstd(vals, axis=0)
    ax.plot(x[:len(mean)], mean, color=color, lw=2.4, label=f"{label} mean")
    ax.fill_between(x[:len(mean)], mean-std, mean+std, color=color, alpha=0.18)
    if ref_da is not None and ref_date is not None:
        rx = date_to_doy(ref_date) if np.asarray(ref_date).max() > 1000 else np.arange(ref_da.sizes["lead_time"])
        ax.plot(rx[:ref_da.sizes["lead_time"]], ref_da.values, color="black", lw=2.0, label="BWCN ref")
    ax.legend(fontsize=8)

for case in inv.loc[inv["has_partial_O3"], "case"]:
    meta = parse_case_name(case)
    o3, date = load_hindcast_o3(case)
    ref_o3, ref_date = load_bwcn_reference_o3(meta["year"])
    u10, u10_date = compute_u60(case, 10)
    u50, u50_date = compute_u60(case, 50)
    fig, axes = plt.subplots(3, 1, figsize=(10.5, 10.5), sharex=True, constrained_layout=True)
    plot_member_envelope(axes[0], o3, date, "O3", "tab:blue", ref_o3, ref_date)
    plot_member_envelope(axes[1], u10, u10_date, "U60N10", "tab:orange")
    plot_member_envelope(axes[2], u50, u50_date, "U60N50", "tab:green")
    for ax in axes:
        ax.axvline(mmdd_to_doy(*init_date_for_case(case)), color="0.35", ls="--", lw=1.0)
        ax.set_xlabel("Day of year")
    axes[0].set_ylabel("O3 (DU)"); axes[1].set_ylabel("m s-1"); axes[2].set_ylabel("m s-1")
    fig.suptitle(f"{case}: O3 and vortex wind evolution")
    case_csv = TAB_DIR / f"02_evolution_{case}_summary.csv"
    row = {"case": case, "year": meta["year"], "init_month": meta["init_month"], "config": meta["config"], "n_members": int(o3.sizes['member']) if o3 is not None else 0}
    if o3 is not None:
        row["O3_ens_min_mean"] = float(o3.min("lead_time", skipna=True).mean("member", skipna=True))
    if u50 is not None:
        fwd = compute_fwd_from_u60n50(u50)
        row["FWD_mean_DOY"] = float(fwd["FWD_DOY"].mean()) if not fwd.empty else np.nan
    pd.DataFrame([row]).to_csv(case_csv, index=False)
    savefig(fig, f"02_evolution_{case}_O3_U60N10_U60N50", notebook="02_multicase_O3_U_evolution.ipynb", scientific_question="Do O3 and vortex wind pathways co-evolve for this case?", variables_windows="O3 60-90N 30-70 hPa; U60N10 and U60N50; init date to May30", interpretation="Member spread or ensemble-mean divergence in U panels provides dynamical context for O3 pathway spread.", caveat="U60 panels require raw U interpolation and may be missing for incomplete cases.", csv_table=case_csv)
    plt.close(fig)
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary_csv = TAB_DIR / "02_multicase_evolution_summary.csv"
summary.to_csv(summary_csv, index=False)
if summary.empty:
    fig = empty_figure("No cases available for evolution plots")
else:
    fig, ax = plt.subplots(figsize=(11, 5.2), constrained_layout=True)
    ax.bar(summary["case"], summary.get("O3_ens_min_mean", np.nan), color="tab:blue")
    ax.set_xticklabels(summary["case"], rotation=60, ha="right")
    ax.set_ylabel("Ensemble mean member-min O3 (DU)")
    ax.set_title("Multi-case O3 depletion overview")
savefig(fig, "02_multicase_evolution_overview_O3_U60N10_U60N50", notebook="02_multicase_O3_U_evolution.ipynb", scientific_question="Which cases have strongest ensemble-mean ozone depletion?", variables_windows="O3 60-90N 30-70 hPa; init date to May30", interpretation="Lower bars identify stronger O3-loss cases for follow-up dynamics and feedback tests.", caveat="Overview compresses full time evolution into one minimum metric.", csv_table=summary_csv)
plt.show()
write_figure_guide()